In [1]:
from pathlib import Path
import pandas as pd

In [2]:
TABLES = Path("../04_outputs/tables")

ml_df = pd.read_csv(TABLES / "baseline_ml_results.csv")
bert_df = pd.read_csv(TABLES / "banglabert_binary_summary.csv")

print("ML results shape:", ml_df.shape)
print("BanglaBERT results shape:", bert_df.shape)

display(ml_df)
display(bert_df)

ML results shape: (3, 7)
BanglaBERT results shape: (6, 13)


,dataset,val_accuracy,val_macro_f1,val_f1_binary,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,0.670823,0.670477,0.659794,0.670823,0.670692,0.664122
1,banglasarc_binary,0.896282,0.887335,0.855586,0.888672,0.880616,0.849604
2,ben_sarc_binary,0.642746,0.642724,0.645511,0.664587,0.664583,0.663537


,model,dataset,split,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,epochs,batch_size,learning_rate,max_length,seed
0,banglabert,banglasarc3_binary,test,0.735661,0.719258,0.773067,0.745192,0.735290,2,8,0.00002,128,42
1,banglabert,banglasarc3_binary,validation,0.763092,0.761787,0.765586,0.763682,0.763091,2,8,0.00002,128,42
2,banglabert,banglasarc_binary,test,0.976562,0.984211,0.954082,0.968912,0.975052,2,8,0.00002,128,42
3,banglabert,banglasarc_binary,validation,0.980431,0.969543,0.979487,0.974490,0.979308,2,8,0.00002,128,42
4,banglabert,ben_sarc_binary,test,0.796412,0.835097,0.738690,0.783940,0.795731,2,8,0.00002,128,42
5,banglabert,ben_sarc_binary,validation,0.792512,0.813545,0.758970,0.785311,0.792278,2,8,0.00002,128,42


In [3]:
# Keep only BanglaBERT test rows
bert_test_df = bert_df[bert_df["split"] == "test"].copy()

bert_test_df = bert_test_df[[
    "dataset", "accuracy", "macro_f1", "f1_binary"
]].rename(columns={
    "accuracy": "test_accuracy",
    "macro_f1": "test_macro_f1",
    "f1_binary": "test_f1_binary"
})

bert_test_df["model"] = "banglabert"
bert_test_df = bert_test_df[[
    "dataset", "model", "test_accuracy", "test_macro_f1", "test_f1_binary"
]]

bert_test_df

,dataset,model,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,banglabert,0.735661,0.735290,0.745192
2,banglasarc_binary,banglabert,0.976562,0.975052,0.968912
4,ben_sarc_binary,banglabert,0.796412,0.795731,0.783940


In [4]:
# Standardize TF-IDF result format
ml_test_df = ml_df[[
    "dataset", "test_accuracy", "test_macro_f1", "test_f1_binary"
]].copy()

ml_test_df["model"] = "tfidf_logreg"

ml_test_df = ml_test_df[[
    "dataset", "model", "test_accuracy", "test_macro_f1", "test_f1_binary"
]]

ml_test_df

,dataset,model,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,tfidf_logreg,0.670823,0.670692,0.664122
1,banglasarc_binary,tfidf_logreg,0.888672,0.880616,0.849604
2,ben_sarc_binary,tfidf_logreg,0.664587,0.664583,0.663537


In [5]:
# Long comparison table
comparison_df = pd.concat([ml_test_df, bert_test_df], ignore_index=True)
comparison_df = comparison_df.sort_values(["dataset", "model"]).reset_index(drop=True)

comparison_df

,dataset,model,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,banglabert,0.735661,0.735290,0.745192
1,banglasarc3_binary,tfidf_logreg,0.670823,0.670692,0.664122
2,banglasarc_binary,banglabert,0.976562,0.975052,0.968912
3,banglasarc_binary,tfidf_logreg,0.888672,0.880616,0.849604
4,ben_sarc_binary,banglabert,0.796412,0.795731,0.783940
5,ben_sarc_binary,tfidf_logreg,0.664587,0.664583,0.663537


In [6]:
# Pivot table using test macro-F1
pivot_df = comparison_df.pivot(
    index="dataset",
    columns="model",
    values="test_macro_f1"
).reset_index()

pivot_df["macro_f1_improvement"] = pivot_df["banglabert"] - pivot_df["tfidf_logreg"]
pivot_df = pivot_df.sort_values("macro_f1_improvement", ascending=False).reset_index(drop=True)

pivot_df

model,dataset,banglabert,tfidf_logreg,macro_f1_improvement
0,ben_sarc_binary,0.795731,0.664583,0.131148
1,banglasarc_binary,0.975052,0.880616,0.094436
2,banglasarc3_binary,0.735290,0.670692,0.064599


In [7]:
comparison_df.to_csv(TABLES / "baseline_model_comparison_long.csv", index=False)
pivot_df.to_csv(TABLES / "baseline_model_comparison_macro_f1.csv", index=False)

print("Saved:")
print(TABLES / "baseline_model_comparison_long.csv")
print(TABLES / "baseline_model_comparison_macro_f1.csv")

Saved:
../04_outputs/tables/baseline_model_comparison_long.csv
../04_outputs/tables/baseline_model_comparison_macro_f1.csv
